In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

In [ ]:
CONTEXTS = ["HCT116", "HEK293T"]
CONTEXT_SHEET = {
    "HCT116": "Table S20",
    "HEK293T": "Table S21",
}
# remove T37 which is the reference point and constent
TEMPS = [
    "T40", "T43", "T46", "T49", "T52", "T55", "T58", "T61", "T64",
]

### Load data and ID mapping

In [ ]:
# Convert symbol to HGNC
hgnc = pd.read_csv(
    "../../data/HGNC/hgnc_complete_set.txt",
    sep="\t", low_memory=False,
)
hgnc = hgnc[hgnc["status"] == "Approved"]
approved_symbols = set(hgnc["symbol"])

# uniprot to gene name
# keep only those resolve to a single gene
uniprot2symbol = {}
for symbol, blob in zip(hgnc["symbol"], hgnc["uniprot_ids"]):
    if not isinstance(blob, str):
        continue
    for accession in blob.split("|"):
        uniprot2symbol.setdefault(accession.strip(), set()).add(symbol)
uniprot2symbol = {
    accession: next(iter(v))
    for accession, v in uniprot2symbol.items()
    if len(v) == 1
}

# old name to current name
# keep only those resolve to a single gene
old2symbol = {}
for column in ["prev_symbol", "alias_symbol"]:
    for symbol, blob in zip(hgnc["symbol"], hgnc[column]):
        if not isinstance(blob, str):
            continue
        for old in blob.split("|"):
            old = old.strip()
            if old == "" or old in approved_symbols:
                continue
            old2symbol.setdefault(old, set()).add(symbol)
old2symbol = {
    old: next(iter(v))
    for old, v in old2symbol.items()
    if len(v) == 1
}

In [ ]:
def load_curves(context, gene_set=None):
    cur = pd.read_excel(
        f"../../data/TPCA/aan0346_Tables_S19_to_S27.xlsx",
        sheet_name=CONTEXT_SHEET[context],
        header=2,
    )
    gene = cur["Accession"].map(uniprot2symbol)
    gene = gene.fillna(cur["GeneName"].map(old2symbol))
    gene = gene.fillna(cur["GeneName"])
    cur = cur.assign(gene=gene)
    cur = cur[cur["gene"] != "-"]
    if gene_set is not None:
        cur = cur[cur["gene"].isin(gene_set)]
    frame = cur.groupby("gene")[TEMPS].mean()
    accession = pd.Series(cur["gene"].values, index=cur["Accession"])
    return frame, accession

In [ ]:
# keep only mod2 genes
ap_hct = set(pd.read_csv("../../result/HEK293T_HCT116/input/BioPlex-HCT116.tsv", sep="\t", index_col=0).index)
ap_hek = set(pd.read_csv("../../result/HEK293T_HCT116/input/BioPlex-HEK293T.tsv", sep="\t", index_col=0).index)
sec_hct = set(pd.read_csv("../../result/HEK293T_HCT116/input/SECMX-HCT116.tsv", sep="\t", index_col=0).index)
sec_hek = set(pd.read_csv("../../result/HEK293T_HCT116/input/SECMX-HEK293T.tsv", sep="\t", index_col=0).index)
gene_set = {}
gene_set["HCT116"] = ap_hct | sec_hct
gene_set["HEK293T"] = ap_hek | sec_hek

In [ ]:
curves = {}
accession_gene = {}
for context in CONTEXTS:
    curves[context], accession_gene[context] = load_curves(context, gene_set[context])

### Calculate score and embedding

In [ ]:
def curve_similarity(X):
    """
    The score from original paper:
    1 / (1 + E)
    where E is Euclidean distance between two melting curves.
    """
    return 1.0 / (1.0 + squareform(pdist(X)))


def standardize(S):
    """
    The Z-score from original paper:
    (score - mu) / sigma
    with mu and sigma over all pairs of the cell line.
    Diagonal is set to zero.
    """
    off = ~np.eye(len(S), dtype=bool)
    Z = (S - S[off].mean()) / S[off].std()
    np.fill_diagonal(Z, 0.0)
    return Z


def kernel_embedding(Z, n_dim):
    """
    Double-center the Z-score matrix and keep the top n_dim
    eigencomponents scaled by the square root of eigenvalues.
    """
    row = Z.mean(axis=1, keepdims=True)
    K = Z - row - row.T + Z.mean()
    values, vectors = np.linalg.eigh(K)
    order = np.argsort(values)[::-1]
    values = values[order]
    vectors = vectors[:, order]
    scale = np.sqrt(values[:n_dim].clip(min=0.0))
    return vectors[:, :n_dim] * scale, values

In [ ]:
os.makedirs("../../result/HEK293T_HCT116_mod3/input/", exist_ok=True)

for context in CONTEXTS:
    X = curves[context][TEMPS].to_numpy(dtype=float)
    Z = standardize(curve_similarity(X))
    emb, values = kernel_embedding(Z, 128)
    df = pd.DataFrame(emb, index=curves[context].index)
    df.to_csv(
        f"../../result/HEK293T_HCT116_mod3/input/TPCA-{context}.tsv",
        sep="\t", index=True, header=True
    )